In [1]:
import pandas as pd
import numpy as np

In [2]:
# 1. Load the raw OPR data
file_path_opr = "/content/OPR-dataset - OPR.csv"
df_opr = pd.read_csv(file_path_opr, header=0, usecols=['Date', 'Rate'])

In [3]:
# 2. View the first 5 rows of CPI data
df_opr.head()

,Date,Rate
0,05 Mar 2026,2.75
1,22 Jan 2026,2.75
2,06 Nov 2025,2.75
3,04 Sep 2025,2.75
4,09 Jul 2025,2.75


In [4]:
# 3. Clean the raw Dates and drop empty rows
df_opr['Date'] = pd.to_datetime(df_opr['Date'], errors='coerce')
df_opr = df_opr.dropna(subset=['Date', 'Rate']).sort_values('Date')

In [5]:
# 4. Extract Year and Month to easily match our master timeline
df_opr['Year'] = df_opr['Date'].dt.year
df_opr['Month'] = df_opr['Date'].dt.month

In [6]:
# 5. View the first 5 rows of OPR data after extracting year and month
df_opr.head()

,Date,Rate,Year,Month
137,2004-05-26,2.7,2004,5
136,2004-08-25,2.7,2004,8
135,2004-11-30,2.7,2004,11
134,2005-02-28,2.7,2005,2
133,2005-05-25,2.7,2005,5


In [7]:
# 6. If BNM somehow changes the rate twice in one month, we just keep the newest one
df_opr_unique = df_opr.drop_duplicates(subset=['Year', 'Month'], keep='last')[['Year', 'Month', 'Rate']]

In [9]:
# 7. Create a Master Timeline (Starting from 2004-05-01 to capture the oldest rate baseline)
timeline = pd.DataFrame({'Date': pd.date_range(start='2004-05-01', end='2026-02-01', freq='MS')})
timeline['Year'] = timeline['Date'].dt.year
timeline['Month'] = timeline['Date'].dt.month

In [10]:
# 8. Merge the BNM OPR changes onto our perfect timeline
merged_opr = pd.merge(timeline, df_opr_unique, on=['Year', 'Month'], how='left')

In [20]:
# 9. View the first and last 5 rows of OPR data after creating timeline
print("First 5 rows")
print(merged_opr.head())
print("\nLast 5 rows")
print(merged_opr.tail())

First 5 rows
        Date  Year  Month  Rate  OPR_Rate
0 2004-05-01  2004      5   2.7       2.7
1 2004-06-01  2004      6   NaN       2.7
2 2004-07-01  2004      7   NaN       2.7
3 2004-08-01  2004      8   2.7       2.7
4 2004-09-01  2004      9   NaN       2.7

Last 5 rows
          Date  Year  Month  Rate  OPR_Rate
257 2025-10-01  2025     10   NaN      2.75
258 2025-11-01  2025     11  2.75      2.75
259 2025-12-01  2025     12   NaN      2.75
260 2026-01-01  2026      1  2.75      2.75
261 2026-02-01  2026      2   NaN      2.75


In [16]:
# 10. Apply the Forward Fill to drag the rate down across empty months
merged_opr['OPR_Rate'] = merged_opr['Rate'].ffill()

In [17]:
# 11. View the first and last 5 rows of OPR data after forward fill
print("first 5 rows")
print(merged_opr.head())
print("\nlast 5 rows")
print(merged_opr.tail())

first 5 rows
        Date  Year  Month  Rate  OPR_Rate
0 2004-05-01  2004      5   2.7       2.7
1 2004-06-01  2004      6   NaN       2.7
2 2004-07-01  2004      7   NaN       2.7
3 2004-08-01  2004      8   2.7       2.7
4 2004-09-01  2004      9   NaN       2.7

last 5 rows
          Date  Year  Month  Rate  OPR_Rate
257 2025-10-01  2025     10   NaN      2.75
258 2025-11-01  2025     11  2.75      2.75
259 2025-12-01  2025     12   NaN      2.75
260 2026-01-01  2026      1  2.75      2.75
261 2026-02-01  2026      2   NaN      2.75


In [18]:
# 12. Filter data exactly for the 2013 to 2026 timeframe
df_opr_clean = merged_opr[(merged_opr['Date'] >= '2013-01-01') & (merged_opr['Date'] <= '2026-01-01')].copy()

In [19]:
# 13. Keep only the essential columns for merging later
df_opr_clean = df_opr_clean[['Date', 'Year', 'Month', 'OPR_Rate']].reset_index(drop=True)

In [21]:
print("OPR Data (First 5 Rows):")
print(df_opr_clean.head())
df_opr_clean
print("\nOPR Data (Last 5 Rows):")
print(df_opr_clean.tail())
print(f"\nTotal Rows: {len(df_opr_clean)}")

OPR Data (First 5 Rows):
        Date  Year  Month  OPR_Rate
0 2013-01-01  2013      1       3.0
1 2013-02-01  2013      2       3.0
2 2013-03-01  2013      3       3.0
3 2013-04-01  2013      4       3.0
4 2013-05-01  2013      5       3.0

OPR Data (Last 5 Rows):
          Date  Year  Month  OPR_Rate
152 2025-09-01  2025      9      2.75
153 2025-10-01  2025     10      2.75
154 2025-11-01  2025     11      2.75
155 2025-12-01  2025     12      2.75
156 2026-01-01  2026      1      2.75

Total Rows: 157
